# Cloud SDKs & Automation — Managing Infrastructure with Python

In [ ]:
# Day 41 — Cloud SDKs & Automation
#
# As a DevOps engineer, you often use Terraform or Bash (Azure CLI / AWS CLI) to manage infrastructure.
# But what if you want to write a script that analyzes CPU usage, decides it needs more power,
# and automatically scales up the VM? Bash gets messy fast.
#
# Python Cloud SDKs (like `azure-mgmt-compute` for Azure or `boto3` for AWS) allow you to interact
# directly with cloud providers natively in Python.
#
# Today we'll cover:
#   1. Authentication (Azure Identity)
#   2. Interacting with Azure SDKs (Listing VMs)
#   3. Modifying Resources (Starting/Stopping VMs)
#   4. AWS Boto3 Equivalent (Brief comparison)
#
# NOTE: You cannot run this live without an Azure subscription and `az login`.
# We will write the code and explain the patterns used in production.
#
# Install: pip install azure-identity azure-mgmt-compute azure-mgmt-resource

### 1. Authentication (Azure Identity)

In [ ]:
# In the old days, you hardcoded a Secret Key in your script. Don't do that!
# Azure provides DefaultAzureCredential. It automatically tries:
# 1. Environment Variables (AZURE_CLIENT_ID, etc.)
# 2. Managed Identity (if running inside an Azure VM or AKS)
# 3. Azure CLI (`az login` cache on your local machine)

azure_auth_code = '''from azure.identity import DefaultAzureCredential

def get_credential():
    print("Acquiring credentials...")
    # This one line handles almost all production and local dev scenarios safely!
    credential = DefaultAzureCredential()
    print("Credentials acquired successfully.")
    return credential

if __name__ == "__main__":
    cred = get_credential()
'''

print("Authentication Pattern:")
print(azure_auth_code)

### 2. Interacting with Azure (Listing VMs)

In [ ]:
# Once authenticated, you pass the credential to a specific "Client".
# Each Azure service has its own client (Compute, Resource, Storage, etc.)

azure_list_code = '''import os
from azure.identity import DefaultAzureCredential
from azure.mgmt.compute import ComputeManagementClient

def list_vms_in_resource_group(subscription_id, resource_group_name):
    # 1. Authenticate
    credential = DefaultAzureCredential()
    
    # 2. Create the Compute Client
    compute_client = ComputeManagementClient(credential, subscription_id)

    # 3. Query Azure
    print(f"Fetching VMs in RG: {resource_group_name}...")
    vm_list = compute_client.virtual_machines.list(resource_group_name)
    
    # 4. Process results
    for vm in vm_list:
        print(f"Found VM: {vm.name}")
        print(f"  Location: {vm.location}")
        print(f"  Size: {vm.hardware_profile.vm_size}")
        print("-" * 20)

# Expected Output:
# Found VM: ml-training-node-01
#   Location: eastus
#   Size: Standard_NC6s_v3 (GPU instance!)
# --------------------
'''

print("Listing VMs Pattern:")
print(azure_list_code)

### 3. Modifying Resources (Cost Saving Script)

In [ ]:
# GPU instances are VERY expensive. 
# Let's write a script an MLOps engineer would use to shut down unused training VMs at night.
# Notice that state changes in Azure are ASYNCHRONOUS long-running operations.

azure_action_code = '''from azure.identity import DefaultAzureCredential
from azure.mgmt.compute import ComputeManagementClient

def stop_gpu_vm(subscription_id, resource_group_name, vm_name):
    credential = DefaultAzureCredential()
    compute_client = ComputeManagementClient(credential, subscription_id)
    
    print(f"Initiating shutdown for {vm_name} to save costs...")
    
    # .begin_deallocate() tells Azure to stop and DEALLOCATE the VM (so you aren't billed)
    # It returns a "poller" object because it takes a few minutes.
    poller = compute_client.virtual_machines.begin_deallocate(resource_group_name, vm_name)
    
    # Wait for the operation to finish
    result = poller.result()
    print(f"{vm_name} is successfully stopped and deallocated.")
'''

print("Modifying Infrastructure Pattern:")
print(azure_action_code)

### 4. Boto3 (The AWS Equivalent)

In [ ]:
# If you use AWS instead of Azure, the concepts are exactly the same.
# AWS uses a library called `boto3`.
#
# Note: Boto3 is slightly older in its design than the Azure SDK, so it uses dictionaries
# heavily rather than strictly typed objects.

boto3_code = '''import boto3

def list_ec2_instances():
    # Boto3 automatically picks up ~/.aws/credentials
    # Creates an EC2 "client"
    ec2 = boto3.client('ec2', region_name='us-east-1')

    # Query AWS
    response = ec2.describe_instances()
    
    # Process the nested JSON-like dictionary
    for reservation in response['Reservations']:
        for instance in reservation['Instances']:
            print(f"Instance ID: {instance['InstanceId']}")
            print(f"  Type: {instance['InstanceType']}")
            print(f"  State: {instance['State']['Name']}")
            print("-" * 20)
'''

print("AWS Boto3 Equivalent:")
print(boto3_code)

In [ ]:
# Practice Questions:
#
# 1. Why do we use `DefaultAzureCredential()` instead of passing a username/password string?
# 2. What is a "poller" in the Azure SDK, and why do functions like `begin_deallocate` return one
#    instead of returning the result immediately?
# 3. Write pseudocode for a Python script that checks the tags of all VMs. If a VM is tagged 
#    with `env: temp` and has been running for > 24 hours, it calls the delete method.

### DevOps / AI-MLOps Practice

In [ ]:
# Exercise 1: The ML Environment Provisioner
# You need to train a massive model. You don't want to leave a GPU VM running 24/7.
# Write a script (combining Typer from Day 39 and Azure SDK from today) that provides a CLI:
#   `python ml_infra.py start --model resnet` -> Starts the VM.
#   `python ml_infra.py stop` -> Deallocates the VM.

# Why this matters for me as a DevOps/MLOps engineer:
# 1. Automation: Terraform is great for day-1 provisioning. Python SDKs are perfect for day-2 operations (auto-scaling, cost-saving shutdowns).
# 2. Cost Control: GPUs cost thousands of dollars a month. Scripting their lifecycle is critical.
# 3. Custom Workflows: You can write a Python script that checks a Jira ticket, approves it, and spins up a VM—something Bash can't do elegantly.